# Download the 384-pixel cache shards from Kaggle to Google Drive

Downloads only the six `pssv27/knee-cache-384-shard-*` outputs that notebook 17 uses.
Each shard has three files: `resolution_receipt_XX.json`, `manifest_train_series_XX.parquet`
and `cache_train_series_XX.npy` (13.4 GiB).

Files go to `MyDrive/knee_cache_384_v1/cache/shardN/`, which is where notebook 17 looks for them,
so its download cell verifies and reuses them instead of downloading again.

Kaggle's API serves the current output of each shard job, which is version 1 (the version
notebook 17 pins). Every manifest and cache file is checked against the SHA-256 in its shard's
receipt before it is saved under its final name.

Allow about **80 GiB of Drive space**. A CPU runtime is enough.

## 1. Mount Google Drive
Authorize access when prompted.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Install the Kaggle API and upload your Kaggle API token
On Kaggle: profile picture → **Settings** → **API** → **Create New Token**. Upload the file it
gives you, either `kaggle.json` or `access_token`. Use the account that owns the private shards
(`pssv27`). The token works like a password, so don't share it.

In [ ]:
%pip install -q "kaggle==2.2.4"
from google.colab import files
uploaded = files.upload()

## 3. Set up the Kaggle directory and permissions
This moves the token into `~/.kaggle`, locks down its permissions, and checks that the account can
see shard 0. It doesn't print the token.

In [ ]:
import os, shutil, subprocess
kaggle_dir = os.path.expanduser('~/.kaggle')
os.makedirs(kaggle_dir, exist_ok=True)
tokens = [name for name in uploaded if name.startswith(('kaggle', 'access_token'))]
if len(tokens) != 1:
    raise RuntimeError(f'Upload exactly one token file (kaggle.json or access_token); got {list(uploaded)}')
target = os.path.join(kaggle_dir, 'kaggle.json' if tokens[0].endswith('.json') else 'access_token')
shutil.move(tokens[0], target)
os.chmod(target, 0o600)
check = subprocess.run(['kaggle', 'kernels', 'status', 'pssv27/knee-cache-384-shard-0'],
                       capture_output=True, text=True)
print(check.stdout + check.stderr)
check.check_returncode()

## 4. Define the downloader
The `kaggle kernels output` command reads a whole file into memory before writing it and shows no
progress, and a 13.4 GiB cache file doesn't fit in Colab's RAM. This cell streams each file straight
to Drive in 8 MiB blocks and shows a progress bar (size, speed, time left).

A file is written as `NAME.partial` first. If the connection drops, it retries and resumes from
where it stopped, and rerunning the next cell also resumes. When a file is complete, its SHA-256
is compared with the receipt, and only then is it renamed to its final name. A complete file
already on Drive is skipped.

In [ ]:
import hashlib, json, time
from pathlib import Path
import requests
from tqdm.auto import tqdm
from kaggle.api.kaggle_api_extended import KaggleApi
from kagglesdk.kernels.types.kernels_api_service import ApiListKernelSessionOutputRequest

DRIVE_CACHE = Path('/content/drive/MyDrive/knee_cache_384_v1')
BLOCK = 8 * 2**20

api = KaggleApi()
api.authenticate()


def output_url(client, shard, name):
    # Signed links expire, so ask for a fresh one on every attempt.
    request = ApiListKernelSessionOutputRequest()
    request.user_name, request.kernel_slug = 'pssv27', f'knee-cache-384-shard-{shard}'
    request.page_size = 100
    response = client.kernels.kernels_api_client.list_kernel_session_output(request)
    urls = {item.file_name: item.url for item in response.files or []}
    if name not in urls:
        raise RuntimeError(f'Shard {shard} has no output named {name}')
    return urls[name]


def remote_size(client, shard, name):
    get = client.http_client()._session.get
    headers = {'Accept-Encoding': 'identity', 'Range': 'bytes=0-0'}
    with get(output_url(client, shard, name), headers=headers, stream=True, timeout=(30, 120)) as response:
        response.raise_for_status()
        if response.status_code == 206:
            return int(response.headers['Content-Range'].rsplit('/', 1)[1])
        return int(response.headers['Content-Length'])


def hash_existing(path, checksum):
    size = path.stat().st_size
    with path.open('rb') as handle, tqdm(total=size, unit='B', unit_scale=True, unit_divisor=1024,
                                          desc=f'checking {path.name}') as bar:
        for block in iter(lambda: handle.read(BLOCK), b''):
            checksum.update(block)
            bar.update(len(block))
    return size


def download(client, shard, name, expected_sha=None, attempts=8):
    directory = DRIVE_CACHE / 'cache' / f'shard{shard}'
    directory.mkdir(parents=True, exist_ok=True)
    target, partial = directory / name, directory / (name + '.partial')
    total = remote_size(client, shard, name)
    if target.exists():
        if target.stat().st_size == total:
            print(f'shard {shard}: {name} already on Drive ({total / 2**30:.2f} GiB), skipping', flush=True)
            return target
        print(f'shard {shard}: removing unfinished {name} from an earlier attempt', flush=True)
        target.unlink()
    get = client.http_client()._session.get
    checksum, hashed = hashlib.sha256(), 0
    for attempt in range(1, attempts + 1):
        offset = partial.stat().st_size if partial.exists() else 0
        if offset > total:
            partial.unlink()
            offset = 0
        if offset != hashed:
            # Resuming a partial written by an earlier run: hash what is already there.
            checksum = hashlib.sha256()
            hashed = hash_existing(partial, checksum) if offset else 0
        headers = {'Accept-Encoding': 'identity'}
        if offset:
            headers['Range'] = f'bytes={offset}-'
        try:
            if offset < total:
                with get(output_url(client, shard, name), headers=headers, stream=True,
                         timeout=(30, 120)) as response:
                    response.raise_for_status()
                    if offset and response.status_code != 206:
                        # The server ignored the resume request; start the file again.
                        offset, checksum, hashed = 0, hashlib.sha256(), 0
                    with partial.open('ab' if offset else 'wb') as handle, \
                            tqdm(total=total, initial=offset, unit='B', unit_scale=True, unit_divisor=1024,
                                 desc=f'shard {shard} {name}') as bar:
                        for block in response.iter_content(chunk_size=BLOCK):
                            handle.write(block)
                            checksum.update(block)
                            hashed += len(block)
                            bar.update(len(block))
            if partial.stat().st_size != total:
                raise requests.exceptions.ChunkedEncodingError('connection closed before the end of the file')
        except (requests.RequestException, OSError) as exc:
            status = getattr(getattr(exc, 'response', None), 'status_code', None)
            if status in (401, 403):
                raise RuntimeError(f'Kaggle denied {name} (HTTP {status}); check the token in step 3') from None
            if attempt == attempts:
                raise RuntimeError(f'{name}: download interrupted; rerun the next cell to resume') from None
            print(f'shard {shard}: {name}: {type(exc).__name__}; retry {attempt}/{attempts - 1}', flush=True)
            time.sleep(min(2 ** attempt, 30))
            continue
        if expected_sha and checksum.hexdigest() != expected_sha:
            partial.unlink()
            raise RuntimeError(f'{name} does not match the checksum in its receipt; rerun the next cell')
        partial.replace(target)
        return target
    raise RuntimeError(f'{name}: download attempts exhausted')

## 5. Download the shards to Drive
This takes a while (~80 GiB). If only some shards fail, set `SHARDS` to those and rerun this cell.
Files that are already complete are skipped, and unfinished ones resume.

In [ ]:
SHARDS = [0, 1, 2, 3, 4, 5]

with api.build_kaggle_client() as client:
    for number, shard in enumerate(SHARDS, 1):
        print(f'=== shard {shard} ({number}/{len(SHARDS)})', flush=True)
        receipt = json.loads(download(client, shard, f'resolution_receipt_{shard:02d}.json').read_text())
        if receipt['state'] != 'complete' or receipt['shard'] != shard:
            raise RuntimeError(f'Shard {shard} receipt is not complete')
        download(client, shard, f'manifest_train_series_{shard:02d}.parquet', receipt['manifest_sha256'])
        download(client, shard, f'cache_train_series_{shard:02d}.npy', receipt['volume_sha256'])
print('Done: all requested shards are on Drive.')

## 6. Check what is on Drive

In [ ]:
for shard in range(6):
    directory = DRIVE_CACHE / 'cache' / f'shard{shard}'
    for name in (f'resolution_receipt_{shard:02d}.json', f'manifest_train_series_{shard:02d}.parquet',
                 f'cache_train_series_{shard:02d}.npy'):
        path, partial = directory / name, directory / (name + '.partial')
        if path.exists():
            size = f'{path.stat().st_size / 2**30:.2f} GiB'
        elif partial.exists():
            size = f'unfinished ({partial.stat().st_size / 2**30:.2f} GiB so far)'
        else:
            size = 'MISSING'
        print(f'shard{shard}/{name}: {size}')